In [1]:
import pandas as pd

# 1. Wczytanie głównej siatki heksagonów
hexagons_df = pd.read_csv('siatka_heksagony.csv', sep=';')

# Oczyszczenie klucza GRID_ID (częsty problem przy eksporcie z GIS)
hexagons_df['GRID_ID'] = hexagons_df['GRID_ID'].astype(str).str.strip()

# 2. Wczytanie danych o hałasie
noise_df = pd.read_csv('halas.csv', sep=';')
noise_df['GRID_ID'] = noise_df['GRID_ID'].astype(str).str.strip()

# 3. Agregacja danych o hałasie
# Wybieramy maksymalną wartość z kolumny 'isov2' dla każdego heksagonu
noise_max_df = noise_df.groupby('GRID_ID', as_index=False)['isov2'].max()

# Zmiana nazwy kolumny na bardziej czytelną (opcjonalnie)
noise_max_df = noise_max_df.rename(columns={'isov2': 'max_halas_dB'})

# 4. Złączenie danych (Left Join - zatrzymujemy wszystkie heksagony z bazy głównej)
master_df = pd.merge(hexagons_df, noise_max_df, on='GRID_ID', how='left')

# (Opcjonalnie) heksagony bez danych o hałasie będą miały wartość NaN. 
# Jeśli chcesz, możesz je wypełnić np. zaniżoną wartością tła (np. 40 dB) lub zostawić puste.
# master_df['max_halas_dB'] = master_df['max_halas_dB'].fillna(40)

print(master_df)

       OBJECTID GRID_ID  Shape_Length    Shape_Area  max_halas_dB
0             1  DV-165    372.241984  10000.002148           NaN
1             2  DW-165    372.241884   9999.996776           NaN
2             3  DX-165    372.241984  10000.002148           NaN
3             4  DZ-165    372.241884   9999.996776           NaN
4             5  EA-165    372.241984  10000.002149           NaN
...         ...     ...           ...           ...           ...
32677     32678    CC-2    372.241984  10000.002149           NaN
32678     32679    CD-2    372.241984  10000.002149           NaN
32679     32680    CE-2    372.241984  10000.002149           NaN
32680     32681    CF-2    372.241984  10000.002149           NaN
32681     32682    CG-2    372.241884   9999.996776           NaN

[32682 rows x 5 columns]


In [2]:
# Wczytanie danych
rowery_df = pd.read_csv('infrastruktura_rower.csv', sep=';')
rowery_df['GRID_ID'] = rowery_df['GRID_ID'].astype(str).str.strip()

# Wielokrotna agregacja: sumujemy sztuki i łączymy unikalne typy w jeden string
rowery_agg = rowery_df.groupby('GRID_ID').agg(
    liczba_infrastruktury_rowerowej=('liczba', 'sum'),
    typy_infrastruktury_rowerowej=('typ', lambda x: ', '.join(x.dropna().unique()))
).reset_index()

# Złączenie z master_df
master_df = pd.merge(master_df, rowery_agg, on='GRID_ID', how='left')

# Czyszczenie braków (0 dla liczb, 'Brak' dla tekstów)
master_df['liczba_infrastruktury_rowerowej'] = master_df['liczba_infrastruktury_rowerowej'].fillna(0).astype(int)
master_df['typy_infrastruktury_rowerowej'] = master_df['typy_infrastruktury_rowerowej'].fillna('Brak')

In [3]:
cas_df = pd.read_csv('CAS.csv', sep=';')
cas_df['GRID_ID'] = cas_df['GRID_ID'].astype(str).str.strip()

# Agregacja nazw obiektów do jednego stringa
cas_agg = cas_df.groupby('GRID_ID').agg(
    nazwy_cas=('Nazwa', lambda x: ', '.join(x.dropna().unique()))
).reset_index()

# Dodanie flagi logicznej dla wygody filtrowania
cas_agg['czy_zawiera_cas'] = 1

# Złączenie z master_df
master_df = pd.merge(master_df, cas_agg, on='GRID_ID', how='left')

# Czyszczenie braków
master_df['czy_zawiera_cas'] = master_df['czy_zawiera_cas'].fillna(0).astype(int)
master_df['nazwy_cas'] = master_df['nazwy_cas'].fillna('Brak')

print(master_df[['GRID_ID', 'czy_zawiera_cas', 'nazwy_cas']].head())

  GRID_ID  czy_zawiera_cas nazwy_cas
0  DV-165                0      Brak
1  DW-165                0      Brak
2  DX-165                0      Brak
3  DZ-165                0      Brak
4  EA-165                0      Brak


In [4]:
# 13. Wczytanie danych o hałasie szynowym (tramwaje/pociągi)
noise_szyny_df = pd.read_csv('halas_szyny.csv', sep=';')
noise_szyny_df['GRID_ID'] = noise_szyny_df['GRID_ID'].astype(str).str.strip()

# 14. Agregacja danych (maksymalna wartość dla heksagonu)
noise_szyny_max_df = noise_szyny_df.groupby('GRID_ID', as_index=False)['isov2'].max()

# Zmiana nazwy kolumny, by odróżnić ją od hałasu ogólnego/drogowego
noise_szyny_max_df = noise_szyny_max_df.rename(columns={'isov2': 'max_halas_szyny_dB'})

# 15. Złączenie z master_df
master_df = pd.merge(master_df, noise_szyny_max_df, on='GRID_ID', how='left')

print(master_df.head())

   OBJECTID GRID_ID  Shape_Length    Shape_Area  max_halas_dB  \
0         1  DV-165    372.241984  10000.002148           NaN   
1         2  DW-165    372.241884   9999.996776           NaN   
2         3  DX-165    372.241984  10000.002148           NaN   
3         4  DZ-165    372.241884   9999.996776           NaN   
4         5  EA-165    372.241984  10000.002149           NaN   

   liczba_infrastruktury_rowerowej typy_infrastruktury_rowerowej nazwy_cas  \
0                                0                          Brak      Brak   
1                                0                          Brak      Brak   
2                                0                          Brak      Brak   
3                                0                          Brak      Brak   
4                                0                          Brak      Brak   

   czy_zawiera_cas  max_halas_szyny_dB  
0                0                 NaN  
1                0                 NaN  
2                

In [5]:
# 16. Wczytanie danych o ciągach rowerowych
ciagi_df = pd.read_csv('ciagi_rowerowe.csv', sep=';')
ciagi_df['GRID_ID'] = ciagi_df['GRID_ID'].astype(str).str.strip()

# 17. Wyciągnięcie unikalnych heksagonów i dodanie flagi
ciagi_unique = ciagi_df[['GRID_ID']].drop_duplicates()
ciagi_unique['czy_zawiera_ciag_rowerowy'] = 1

# 18. Złączenie z główną tabelą master_df
master_df = pd.merge(master_df, ciagi_unique, on='GRID_ID', how='left')

# 19. Zamiana braków na zera i rzutowanie na int
master_df['czy_zawiera_ciag_rowerowy'] = master_df['czy_zawiera_ciag_rowerowy'].fillna(0).astype(int)

print(master_df.head())

   OBJECTID GRID_ID  Shape_Length    Shape_Area  max_halas_dB  \
0         1  DV-165    372.241984  10000.002148           NaN   
1         2  DW-165    372.241884   9999.996776           NaN   
2         3  DX-165    372.241984  10000.002148           NaN   
3         4  DZ-165    372.241884   9999.996776           NaN   
4         5  EA-165    372.241984  10000.002149           NaN   

   liczba_infrastruktury_rowerowej typy_infrastruktury_rowerowej nazwy_cas  \
0                                0                          Brak      Brak   
1                                0                          Brak      Brak   
2                                0                          Brak      Brak   
3                                0                          Brak      Brak   
4                                0                          Brak      Brak   

   czy_zawiera_cas  max_halas_szyny_dB  czy_zawiera_ciag_rowerowy  
0                0                 NaN                          0  
1   

In [6]:
# 20. Wczytanie izochron
izochrony_df = pd.read_csv('izochrony_km.csv', sep=';')
izochrony_df['GRID_ID'] = izochrony_df['GRID_ID'].astype(str).str.strip()

# 21. Filtrujemy tylko obszary do 100 metrów
# Zależnie od tego, jak zapisano typ w CSV, może to być 100, 100.0 lub '100'
izochrony_100m = izochrony_df[izochrony_df['odległość'] == 100.0].copy()

# 22. Agregacja przystanków
# Zbieramy unikalne nazwy przystanków dla heksagonu i łączymy je w jeden ciąg tekstowy (string)
przystanki_agg = (
    izochrony_100m.groupby('GRID_ID')['nazwa zespołu przystankowego']
    .unique()
    .apply(lambda x: ', '.join(x))
    .reset_index()
)
przystanki_agg = przystanki_agg.rename(columns={'nazwa zespołu przystankowego': 'przystanki_do_100m'})

# Dodajemy od razu flagę logiczną ułatwiającą filtrowanie w aplikacji
przystanki_agg['czy_przystanek_do_100m'] = 1

# 23. Złączenie z master_df
master_df = pd.merge(master_df, przystanki_agg, on='GRID_ID', how='left')

# 24. Wypełnienie braków danych dla heksagonów bez przystanku
master_df['czy_przystanek_do_100m'] = master_df['czy_przystanek_do_100m'].fillna(0).astype(int)
master_df['przystanki_do_100m'] = master_df['przystanki_do_100m'].fillna('Brak')

print(master_df[['GRID_ID', 'czy_przystanek_do_100m', 'przystanki_do_100m']].head())

  GRID_ID  czy_przystanek_do_100m przystanki_do_100m
0  DV-165                       0               Brak
1  DW-165                       0               Brak
2  DX-165                       0               Brak
3  DZ-165                       0               Brak
4  EA-165                       0               Brak


In [7]:
# --- 1. Placówki ochrony zdrowia ---
zdrowie_df = pd.read_csv('zdrowie.csv', sep=';')
zdrowie_df['GRID_ID'] = zdrowie_df['GRID_ID'].astype(str).str.strip()

zdrowie_agg = zdrowie_df.groupby('GRID_ID').agg(
    nazwy_zdrowie=('nazwa', lambda x: ', '.join(x.dropna().unique()))
).reset_index()

zdrowie_agg['czy_zawiera_zdrowie'] = 1

master_df = pd.merge(master_df, zdrowie_agg, on='GRID_ID', how='left')
master_df['czy_zawiera_zdrowie'] = master_df['czy_zawiera_zdrowie'].fillna(0).astype(int)
master_df['nazwy_zdrowie'] = master_df['nazwy_zdrowie'].fillna('Brak')


# --- 2. Instytucje ---
instytucje_df = pd.read_csv('instytucje_razem.csv', sep=';')
instytucje_df['GRID_ID'] = instytucje_df['GRID_ID'].astype(str).str.strip()

instytucje_agg = instytucje_df.groupby('GRID_ID').agg(
    nazwy_instytucje=('nazwa', lambda x: ', '.join(x.dropna().unique()))
).reset_index()

instytucje_agg['czy_zawiera_instytucje'] = 1

master_df = pd.merge(master_df, instytucje_agg, on='GRID_ID', how='left')
master_df['czy_zawiera_instytucje'] = master_df['czy_zawiera_instytucje'].fillna(0).astype(int)
master_df['nazwy_instytucje'] = master_df['nazwy_instytucje'].fillna('Brak')

print(master_df[['GRID_ID', 'czy_zawiera_zdrowie', 'nazwy_zdrowie', 'czy_zawiera_instytucje', 'nazwy_instytucje']].head())

  GRID_ID  czy_zawiera_zdrowie nazwy_zdrowie  czy_zawiera_instytucje  \
0  DV-165                    0          Brak                       0   
1  DW-165                    0          Brak                       0   
2  DX-165                    0          Brak                       0   
3  DZ-165                    0          Brak                       0   
4  EA-165                    0          Brak                       0   

  nazwy_instytucje  
0             Brak  
1             Brak  
2             Brak  
3             Brak  
4             Brak  


In [8]:
# --- 1. Toalety publiczne ---
toalety_df = pd.read_csv('toalety.csv', sep=';')
toalety_df['GRID_ID'] = toalety_df['GRID_ID'].astype(str).str.strip()

toalety_unique = toalety_df[['GRID_ID']].drop_duplicates()
toalety_unique['czy_zawiera_toalete'] = 1

master_df = pd.merge(master_df, toalety_unique, on='GRID_ID', how='left')
master_df['czy_zawiera_toalete'] = master_df['czy_zawiera_toalete'].fillna(0).astype(int)

# --- 2. Policja ---
policja_df = pd.read_csv('policja.csv', sep=';')
policja_df['GRID_ID'] = policja_df['GRID_ID'].astype(str).str.strip()

policja_unique = policja_df[['GRID_ID']].drop_duplicates()
policja_unique['czy_zawiera_policje'] = 1

master_df = pd.merge(master_df, policja_unique, on='GRID_ID', how='left')
master_df['czy_zawiera_policje'] = master_df['czy_zawiera_policje'].fillna(0).astype(int)

print(master_df[['GRID_ID', 'czy_zawiera_toalete', 'czy_zawiera_policje']].head())

  GRID_ID  czy_zawiera_toalete  czy_zawiera_policje
0  DV-165                    0                    0
1  DW-165                    0                    0
2  DX-165                    0                    0
3  DZ-165                    0                    0
4  EA-165                    0                    0


In [9]:
# --- 1. Pitniki wodne (tylko flaga) ---
pitniki_df = pd.read_csv('pitniki.csv', sep=';')
pitniki_df['GRID_ID'] = pitniki_df['GRID_ID'].astype(str).str.strip()

pitniki_unique = pitniki_df[['GRID_ID']].drop_duplicates()
pitniki_unique['czy_zawiera_pitnik'] = 1

master_df = pd.merge(master_df, pitniki_unique, on='GRID_ID', how='left')
master_df['czy_zawiera_pitnik'] = master_df['czy_zawiera_pitnik'].fillna(0).astype(int)

# --- 2. MOPS (flaga + nazwy) ---
mops_df = pd.read_csv('mops.csv', sep=';')
mops_df['GRID_ID'] = mops_df['GRID_ID'].astype(str).str.strip()

mops_agg = mops_df.groupby('GRID_ID').agg(
    nazwy_mops=('NAZWA', lambda x: ', '.join(x.dropna().unique()))
).reset_index()

mops_agg['czy_zawiera_mops'] = 1

master_df = pd.merge(master_df, mops_agg, on='GRID_ID', how='left')
master_df['czy_zawiera_mops'] = master_df['czy_zawiera_mops'].fillna(0).astype(int)
master_df['nazwy_mops'] = master_df['nazwy_mops'].fillna('Brak')

print(master_df[['GRID_ID', 'czy_zawiera_pitnik', 'czy_zawiera_mops', 'nazwy_mops']].head())

  GRID_ID  czy_zawiera_pitnik  czy_zawiera_mops nazwy_mops
0  DV-165                   0                 0       Brak
1  DW-165                   0                 0       Brak
2  DX-165                   0                 0       Brak
3  DZ-165                   0                 0       Brak
4  EA-165                   0                 0       Brak


In [10]:
# 1. Wczytanie danych MPZP
mpzp_df = pd.read_csv('mpzp.csv', sep=';')
mpzp_df['GRID_ID'] = mpzp_df['GRID_ID'].astype(str).str.strip()

# NAPRAWA: Wymuszamy konwersję na typ daty (errors='coerce' zamieni błędne formaty/puste komórki na NaT)
mpzp_df['data_obowi'] = pd.to_datetime(mpzp_df['data_obowi'], errors='coerce')

# Zabezpieczenie kolumny z tekstem (gdyby tam też trafił się NaN zmieszany z tekstem)
mpzp_df['opis_oznac'] = mpzp_df['opis_oznac'].astype(str)

# 2. Agregacja pod dymek informacyjny w aplikacji
mpzp_agg = mpzp_df.groupby('GRID_ID').agg(
    mpzp_przeznaczenia=('opis_oznac', lambda x: ', '.join(x.replace('nan', pd.NA).dropna().unique())),
    mpzp_rok_planu=('data_obowi', 'max') # Teraz max() zadziała bez błędu
).reset_index()

# 3. Złączenie z master_df
master_df = pd.merge(master_df, mpzp_agg, on='GRID_ID', how='left')

# Formatujemy datę z powrotem na ładny tekst (np. wyciągamy sam rok lub pełną datę)
master_df['mpzp_rok_planu'] = master_df['mpzp_rok_planu'].dt.strftime('%Y-%m-%d')

# 4. Jasna informacja o braku planu dla pustych komórek
master_df['mpzp_przeznaczenia'] = master_df['mpzp_przeznaczenia'].fillna('Brak planu')
master_df['mpzp_rok_planu'] = master_df['mpzp_rok_planu'].fillna('Brak')

print(master_df[['GRID_ID', 'mpzp_przeznaczenia', 'mpzp_rok_planu']].head())

  GRID_ID                                 mpzp_przeznaczenia mpzp_rok_planu
0  DV-165  1 R(Z) DO 6 R(Z) - Tereny rolnicze w postaci z...           Brak
1  DW-165                                         Brak planu           Brak
2  DX-165                                         Brak planu           Brak
3  DZ-165                                         Brak planu           Brak
4  EA-165  1 WS i 2 WS - Tereny wód powierzchniowych śród...           Brak


In [11]:
import numpy as np
# 1. Wczytanie danych
edukacja_df = pd.read_csv('edukacja.csv', sep=';')
edukacja_df['GRID_ID'] = edukacja_df['GRID_ID'].astype(str).str.strip()

# Zamiana pustych ciągów znaków (lub samych spacji) na NaN w interesujących nas kolumnach
for col in ['kategoria_edukacja', 'organizacja', 'name']:
    if col in edukacja_df.columns:
        edukacja_df[col] = edukacja_df[col].replace(r'^\s*$', np.nan, regex=True)

# 2. Filtrowanie danych zgodnie z wytycznymi:
# - Musi być kategoria edukacji
# - Musi być przynajmniej jedna nazwa (organizacja LUB name nie może być puste)
edukacja_clean = edukacja_df.dropna(subset=['kategoria_edukacja']).copy()
edukacja_clean = edukacja_clean.dropna(subset=['organizacja', 'name'], how='all')

# 3. Scalamy nazwy: najpierw bierzemy 'organizacja', a jeśli jest pusta, podstawiamy 'name'
edukacja_clean['nazwa_ostateczna'] = edukacja_clean['organizacja'].fillna(edukacja_clean['name'])

# 4. Agregacja na poziom heksagonu
edukacja_agg = edukacja_clean.groupby('GRID_ID').agg(
    kategorie_edukacji=('kategoria_edukacja', lambda x: ', '.join(x.dropna().unique())),
    nazwy_edukacja=('nazwa_ostateczna', lambda x: ', '.join(x.dropna().unique()))
).reset_index()

# Dodanie flagi logicznej
edukacja_agg['czy_zawiera_edukacje'] = 1

# 5. Złączenie z naszą główną tabelą master_df
master_df = pd.merge(master_df, edukacja_agg, on='GRID_ID', how='left')

# 6. Uzupełnienie braków (NaN) dla heksagonów bez edukacji
master_df['czy_zawiera_edukacje'] = master_df['czy_zawiera_edukacje'].fillna(0).astype(int)
master_df['kategorie_edukacji'] = master_df['kategorie_edukacji'].fillna('Brak')
master_df['nazwy_edukacja'] = master_df['nazwy_edukacja'].fillna('Brak')

print(master_df[['GRID_ID', 'czy_zawiera_edukacje', 'kategorie_edukacji', 'nazwy_edukacja']].head())

  GRID_ID  czy_zawiera_edukacje kategorie_edukacji nazwy_edukacja
0  DV-165                     0               Brak           Brak
1  DW-165                     0               Brak           Brak
2  DX-165                     0               Brak           Brak
3  DZ-165                     0               Brak           Brak
4  EA-165                     0               Brak           Brak


In [12]:
# 1. Wczytanie danych o sporcie
sport_df = pd.read_csv('sport.csv', sep=';')
sport_df['GRID_ID'] = sport_df['GRID_ID'].astype(str).str.strip()

# 2. Zamiana pustych ciągów znaków (lub samych spacji) na NaN
for col in ['NAZWA', 'rodzaj_obiektu', 'DYSCYPLINA']:
    if col in sport_df.columns:
        sport_df[col] = sport_df[col].replace(r'^\s*$', np.nan, regex=True)

# 3. Pomijamy wiersze, w których KTÓRAKOLWIEK z tych trzech kluczowych kolumn jest pusta
sport_clean = sport_df.dropna(subset=['NAZWA', 'rodzaj_obiektu', 'DYSCYPLINA']).copy()

# 4. Agregacja na poziom heksagonu (osobno nazwy, rodzaje i dyscypliny)
sport_agg = sport_clean.groupby('GRID_ID').agg(
    nazwy_sport=('NAZWA', lambda x: ', '.join(x.dropna().unique())),
    rodzaje_sportu=('rodzaj_obiektu', lambda x: ', '.join(x.dropna().unique())),
    dyscypliny_sportowe=('DYSCYPLINA', lambda x: ', '.join(x.dropna().unique()))
).reset_index()

# Dodanie flagi logicznej
sport_agg['czy_zawiera_sport'] = 1

# 5. Złączenie z naszą główną tabelą master_df
master_df = pd.merge(master_df, sport_agg, on='GRID_ID', how='left')

# 6. Uzupełnienie braków (NaN) dla heksagonów bez sportu
master_df['czy_zawiera_sport'] = master_df['czy_zawiera_sport'].fillna(0).astype(int)
master_df['nazwy_sport'] = master_df['nazwy_sport'].fillna('Brak')
master_df['rodzaje_sportu'] = master_df['rodzaje_sportu'].fillna('Brak')
master_df['dyscypliny_sportowe'] = master_df['dyscypliny_sportowe'].fillna('Brak')

print(master_df[['GRID_ID', 'czy_zawiera_sport', 'nazwy_sport', 'dyscypliny_sportowe']].head())

  GRID_ID  czy_zawiera_sport nazwy_sport dyscypliny_sportowe
0  DV-165                  0        Brak                Brak
1  DW-165                  0        Brak                Brak
2  DX-165                  0        Brak                Brak
3  DZ-165                  0        Brak                Brak
4  EA-165                  0        Brak                Brak


In [13]:
# 1. Wczytanie danych o dostępności dla niepełnosprawnych
niepelnosprawni_df = pd.read_csv('niepelnosprawni.csv', sep=';')
niepelnosprawni_df['GRID_ID'] = niepelnosprawni_df['GRID_ID'].astype(str).str.strip()

# 2. Konwersja kolumny score na liczby (błędy/puste pola zamieniamy na NaN)
niepelnosprawni_df['score_num'] = pd.to_numeric(niepelnosprawni_df['Score'], errors='coerce')

# 3. Agregacja na poziom heksagonu (bierzemy średnią wartość score, jeśli jest wiele obiektów)
niepelnosprawni_agg = niepelnosprawni_df.groupby('GRID_ID').agg(
    score_niepelnosprawni=('score_num', 'mean')
).reset_index()

# 4. Złączenie z naszą główną tabelą master_df
master_df = pd.merge(master_df, niepelnosprawni_agg, on='GRID_ID', how='left')

# 5. Formatowanie: wartości numeryczne zaokrąglamy do 1 miejsca po przecinku, a braki zamieniamy na 'Brak danych'
master_df['score_niepelnosprawni'] = master_df['score_niepelnosprawni'].apply(
    lambda x: f"{x:.1f}" if pd.notna(x) else 'Brak danych'
)

print(master_df[['GRID_ID', 'score_niepelnosprawni']].head())

  GRID_ID score_niepelnosprawni
0  DV-165           Brak danych
1  DW-165           Brak danych
2  DX-165           Brak danych
3  DZ-165           Brak danych
4  EA-165           Brak danych


In [14]:
# 1. Wczytanie rejestru cen
ceny_df = pd.read_csv('rejestr_cen.csv', sep=';')
ceny_df['GRID_ID'] = ceny_df['GRID_ID'].astype(str).str.strip()
ceny_df['data_zaw_month'] = ceny_df['data_zaw_month'].astype(str).str.strip()

# 2. Bezpieczna konwersja kolumn numerycznych (gdyby w pliku były przecinki zamiast kropek)
for col in ['sr_cena', 'sr_pow', 'sr_cena_m2']:
    if col in ceny_df.columns:
        if ceny_df[col].dtype == object:
            ceny_df[col] = ceny_df[col].astype(str).str.replace(',', '.', regex=False)
        ceny_df[col] = pd.to_numeric(ceny_df[col], errors='coerce')

# 3. Sortujemy dane rosnąco po ID, ale malejąco po dacie (najnowsza data na samej górze grupy)
ceny_sorted = ceny_df.sort_values(by=['GRID_ID', 'data_zaw_month'], ascending=[True, False])

# 4. Zostawiamy tylko pierwszy (najświeższy) rekord dla każdego heksagonu
ceny_latest = ceny_sorted.drop_duplicates(subset=['GRID_ID'], keep='first').copy()

# Zmiana nazw kolumn, żeby były unikalne i intuicyjne w master_df
ceny_latest = ceny_latest[['GRID_ID', 'sr_cena', 'sr_pow', 'sr_cena_m2', 'data_zaw_month']].rename(columns={
    'sr_cena': 'cena_sr_calkowita',
    'sr_pow': 'cena_sr_powierzchnia',
    'sr_cena_m2': 'cena_sr_za_m2',
    'data_zaw_month': 'cena_data_aktualnosci'
})

# 5. Złączenie z główną tabelą master_df
master_df = pd.merge(master_df, ceny_latest, on='GRID_ID', how='left')

# Uwaga: kolumn numerycznych (cena_sr_za_m2) NIE wypełniamy zerami ani napisem 'Brak danych', 
# zostawiamy NaN, żeby biblioteki mapowe (np. PyDeck/Folium) wiedziały, że dla tych heksagonów 
# nie ma transakcji i nie rysowały ich na czarno/zielono.

print(master_df[['GRID_ID', 'cena_sr_za_m2', 'cena_data_aktualnosci']].head(10))

  GRID_ID  cena_sr_za_m2 cena_data_aktualnosci
0  DV-165            NaN                   NaN
1  DW-165            NaN                   NaN
2  DX-165            NaN                   NaN
3  DZ-165            NaN                   NaN
4  EA-165            NaN                   NaN
5  EB-165            NaN                   NaN
6  DT-164            NaN                   NaN
7  DU-164            NaN                   NaN
8  DV-164            NaN                   NaN
9  DW-164            NaN                   NaN


In [15]:
# 1. Wczytanie danych o wietrze / wentylacji
wiatr_df = pd.read_csv('wiatr_monitAIR.csv', sep=';')
wiatr_df['GRID_ID'] = wiatr_df['GRID_ID'].astype(str).str.strip()
# 2. Czyszczenie i konwersja kolumn numerycznych (zamiana przecinków na kropki, jeśli występują)
for col in ['wentyl_v4m', 'wentyl_pop']:
    if col in wiatr_df.columns:
        if wiatr_df[col].dtype == object:
            wiatr_df[col] = wiatr_df[col].str.replace(',', '.', regex=False)
        wiatr_df[col] = pd.to_numeric(wiatr_df[col], errors='coerce')

# 3. Agregacja na poziom heksagonu (bierzemy średnią, jeśli jest kilka rekordów na grid)
wiatr_agg = wiatr_df.groupby('GRID_ID').agg(
    wentyl_predkosc_v4m=('wentyl_v4m', 'mean'),
    wentyl_potencjal=('wentyl_pop', 'mean')
).reset_index()

# 4. Złączenie z główną tabelą master_df
master_df = pd.merge(master_df, wiatr_agg, on='GRID_ID', how='left')

# 5. Podgląd wyników
print(master_df[['GRID_ID', 'wentyl_predkosc_v4m', 'wentyl_potencjal']].head())

  GRID_ID  wentyl_predkosc_v4m  wentyl_potencjal
0  DV-165             2.595464          2.595464
1  DW-165                  NaN               NaN
2  DX-165             3.097860          3.097860
3  DZ-165             2.719868          2.719868
4  EA-165             2.408418          2.408418


In [16]:
import pandas as pd
import numpy as np

# 1. Wczytanie danych o zielonych korytarzach 
korytarze_df = pd.read_csv('zielone_korytarze.csv', sep=';')

# Upewniamy się, że GRID_ID jest oczyszczony
korytarze_df['GRID_ID'] = korytarze_df['GRID_ID'].astype(str).str.strip()

# 2. Czyszczenie i obsługa pustych wartości w kolumnach tekstowych
korytarze_df['NAZWA'] = korytarze_df['NAZWA'].replace(r'^\s*$', np.nan, regex=True).fillna('Brak nazwy szlaku')
korytarze_df['STATUS'] = korytarze_df['STATUS'].replace(r'^\s*$', np.nan, regex=True).fillna('Brak statusu')

# Tworzymy czytelny opis pod dymek w aplikacji
korytarze_df['opis_korytarza'] = korytarze_df['NAZWA'] + ' (' + korytarze_df['STATUS'] + ')'

# 3. Agregacja na poziom heksagonu (jeśli przez jeden heksagon przechodzi kilka korytarzy/odcinków)
korytarze_agg = korytarze_df.groupby('GRID_ID').agg(
    nazwy_korytarzy_zieleni=('opis_korytarza', lambda x: ', '.join(x.dropna().unique()))
).reset_index()

# Dodanie flagi logicznej
korytarze_agg['czy_zielony_korytarz'] = 1

# 4. Złączenie z naszą główną tabelą master_df
master_df = pd.merge(master_df, korytarze_agg, on='GRID_ID', how='left')

# 5. Uzupełnienie braków danych dla heksagonów leżących poza korytarzami
master_df['czy_zielony_korytarz'] = master_df['czy_zielony_korytarz'].fillna(0).astype(int)
master_df['nazwy_korytarzy_zieleni'] = master_df['nazwy_korytarzy_zieleni'].fillna('Brak')

print("Sukces! Dodano zielone korytarze. Aktualna liczba kolumn w master_df:", master_df.shape[1])

Sukces! Dodano zielone korytarze. Aktualna liczba kolumn w master_df: 40


# Dodaj geometrię i kompresuj do parquet

In [17]:
import geopandas as gpd
import pandas as pd

# 1. Wczytanie geometrii heksagonów z pliku GeoJSON
# GeoPandas automatycznie sparsuje współrzędne do obiektu geometrycznego (Polygon)
gdf_hex = gpd.read_file('siatka_heksagony.geojson')

# Upewniamy się, że klucz łączenia ma ten sam format (string bez spacji)
gdf_hex['GRID_ID'] = gdf_hex['GRID_ID'].astype(str).str.strip()
master_df['GRID_ID'] = master_df['GRID_ID'].astype(str).str.strip()

# 2. Łączymy naszą bogatą tabelę master_df z geometrią GeoJSON po GRID_ID
# Używamy right lub inner join, żeby zatrzymać tylko te heksagony, które mają geometrię
final_gdf = pd.merge(gdf_hex, master_df, on='GRID_ID', how='inner')

# 3. Zapis do superszybkiego formatu Geopackage lub Parquet
# Format .parquet (lub .gpkg) ładuje się w Pythonie kilkadziesiąt razy szybciej niż CSV!
final_gdf.to_parquet('krakow_master_grid.parquet')

print(f"Sukces! Połączono geometrie z danymi. Łączna liczba heksagonów w modelu: {len(final_gdf)}")

Sukces! Połączono geometrie z danymi. Łączna liczba heksagonów w modelu: 32682
